In [1]:
!pip install -q transformers accelerate scikit-learn pillow matplotlib pandas numpy tqdm


In [2]:
import os, json, math, random, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/data_aware_chart_retrieval')
for d in ['data/tables','data/charts','data/queries','data/counterfactuals','data/metadata','models','results']:
    (PROJECT_ROOT/d).mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)


/content/data_aware_chart_retrieval


## Stage 2 — Synthetic table generator + property extraction


In [3]:
PATTERNS = ['rising','falling','flat','peak','valley','irregular']

def generate_years():
    n=random.randint(4,8); start=random.randint(2000,2018)
    return list(range(start,start+n))

def generate_rising(n):
    while True:
        vals=[random.randint(5,35)]
        for _ in range(n-1): vals.append(vals[-1]+random.randint(4,12))
        if max(vals)<=100: return vals

def generate_falling(n):
    while True:
        vals=[random.randint(65,100)]
        for _ in range(n-1): vals.append(vals[-1]-random.randint(4,12))
        if min(vals)>=5: return vals

def generate_flat(n):
    base=random.randint(20,80)
    return [int(np.clip(base+random.randint(-2,2),5,100)) for _ in range(n)]

def generate_peak(n):
    k=random.randint(1,n-2); start=random.randint(5,25); vals=[start]
    for _ in range(k): vals.append(vals[-1]+random.randint(7,13))
    for _ in range(k+1,n): vals.append(vals[-1]-random.randint(7,13))
    if min(vals)<5 or max(vals)>100: return generate_peak(n)
    return vals

def generate_valley(n):
    k=random.randint(1,n-2); start=random.randint(75,100); vals=[start]
    for _ in range(k): vals.append(vals[-1]-random.randint(7,13))
    for _ in range(k+1,n): vals.append(vals[-1]+random.randint(7,13))
    if min(vals)<5 or max(vals)>100: return generate_valley(n)
    return vals

def generate_irregular(n):
    while True:
        vals=[random.randint(5,100) for _ in range(n)]
        if not all(vals[i]<vals[i+1] for i in range(n-1)) and not all(vals[i]>vals[i+1] for i in range(n-1)):
            return vals

def is_strictly_increasing(v): return all(v[i]<v[i+1] for i in range(len(v)-1))
def is_strictly_decreasing(v): return all(v[i]>v[i+1] for i in range(len(v)-1))
def calculate_changes(v): return [v[i+1]-v[i] for i in range(len(v)-1)]
def calculate_percentage_changes(v): return [100*(v[i+1]-v[i])/v[i] for i in range(len(v)-1)]

def detect_peak(years,v):
    i=int(np.argmax(v)); ok=0<i<len(v)-1 and v[i]>v[i-1] and v[i]>v[i+1]
    return {'has_peak':ok,'peak_year':years[i] if ok else None,'peak_value':v[i] if ok else None}

def detect_valley(years,v):
    i=int(np.argmin(v)); ok=0<i<len(v)-1 and v[i]<v[i-1] and v[i]<v[i+1]
    return {'has_valley':ok,'valley_year':years[i] if ok else None,'valley_value':v[i] if ok else None}

def extract_properties(table):
    y,v=table['years'],table['values']; imax=int(np.argmax(v)); imin=int(np.argmin(v))
    p={'num_points':len(v),'start_year':y[0],'end_year':y[-1],'start_value':v[0],'end_value':v[-1],
       'max_value':max(v),'max_year':y[imax],'min_value':min(v),'min_year':y[imin],
       'strictly_increasing':is_strictly_increasing(v),'strictly_decreasing':is_strictly_decreasing(v),
       'changes':calculate_changes(v),'percentage_changes':calculate_percentage_changes(v)}
    p.update(detect_peak(y,v)); p.update(detect_valley(y,v)); return p

def generate_table(table_id):
    years=generate_years(); n=len(years); pattern=random.choice(PATTERNS)
    fn={'rising':generate_rising,'falling':generate_falling,'flat':generate_flat,'peak':generate_peak,'valley':generate_valley,'irregular':generate_irregular}[pattern]
    table={'table_id':table_id,'years':years,'values':fn(n),'pattern':pattern}
    table['properties']=extract_properties(table); return table

def table_text(table):
    return ' | '.join(f'{y}: {v}' for y,v in zip(table['years'],table['values']))

print(generate_table('demo'))


{'table_id': 'demo', 'years': [2000, 2001, 2002, 2003], 'values': [99, 18, 91, 99], 'pattern': 'irregular', 'properties': {'num_points': 4, 'start_year': 2000, 'end_year': 2003, 'start_value': 99, 'end_value': 99, 'max_value': 99, 'max_year': 2000, 'min_value': 18, 'min_year': 2001, 'strictly_increasing': False, 'strictly_decreasing': False, 'changes': [-81, 73, 8], 'percentage_changes': [-81.81818181818181, 405.55555555555554, 8.791208791208792], 'has_peak': False, 'peak_year': None, 'peak_value': None, 'has_valley': True, 'valley_year': 2001, 'valley_value': 18}}


## Stage 3 — Controlled chart renderer


In [4]:
CHART_TYPES=['bar','line']; COLORS=['blue','red','green','orange','purple']
ALL_STYLES=[{'chart_type':t,'color':c} for t in CHART_TYPES for c in COLORS]

def select_random_styles(num_styles=4):
    bars=[s for s in ALL_STYLES if s['chart_type']=='bar']; lines=[s for s in ALL_STYLES if s['chart_type']=='line']
    selected=[random.choice(bars),random.choice(lines)]
    selected += random.sample([s for s in ALL_STYLES if s not in selected],num_styles-2)
    return selected

def render_chart(table, chart_type, color, output_path, show=False):
    fig,ax=plt.subplots(figsize=(2.24,2.24),dpi=100)
    if chart_type=='bar': ax.bar(table['years'],table['values'],color=color)
    else: ax.plot(table['years'],table['values'],color=color,marker='o',linewidth=2)
    ax.set_ylim(0,115); ax.set_xticks(table['years']); ax.set_title(''); fig.tight_layout()
    fig.savefig(output_path,dpi=100);
    if show: plt.show()
    plt.close(fig)

def generate_charts_for_table(table,num_styles=4):
    out=[]
    for i,s in enumerate(select_random_styles(num_styles)):
        cid=f"{table['table_id']}_chart_{i}"; path=PROJECT_ROOT/'data/charts'/f'{cid}.png'
        render_chart(table,s['chart_type'],s['color'],path)
        out.append({'chart_id':cid,'table_id':table['table_id'],'chart_type':s['chart_type'],'color':s['color'],'path':str(path)})
    return out


## Stage 4 — Standard shuffled hard negatives


In [5]:
def shuffle_values(values):
    while True:
        x=list(values); random.shuffle(x)
        if x!=list(values): return x

def create_shuffled_negative(table):
    neg={'table_id':table['table_id']+'_shuffled','source_table_id':table['table_id'],'years':table['years'].copy(),
         'values':shuffle_values(table['values']),'pattern':'shuffled_negative','negative_type':'shuffle'}
    neg['properties']=extract_properties(neg); return neg

def data_difference(a,b): return float(np.mean(np.abs(np.array(a,float)-np.array(b,float))))


## Stage 5 — Natural-language query generator + executable evaluator


In [6]:
INC=['Find a chart where the values increase every year.','Which chart shows a steady increase over time?','Retrieve the chart whose values rise each year.']
DEC=['Find a chart where the values decrease every year.','Which chart shows a steady decline over time?','Retrieve the chart whose values fall each year.']
MAXT=['Find the chart whose highest value occurs in {year}.','Which chart reaches its maximum in {year}?']
MINT=['Find the chart whose lowest value occurs in {year}.','Which chart reaches its minimum in {year}?']
EXACT=['Find the chart where the value in {year} is {value}.','Which chart shows a value of {value} in {year}?']
ENDHI=['Find a chart whose final value is greater than its first value.','Which chart ends higher than it begins?']
ENDLO=['Find a chart whose final value is lower than its first value.','Which chart ends lower than it begins?']

def qrec(qid,tid,qt,text,prop,val=None,year=None):
    return {'query_id':qid,'table_id':tid,'query_type':qt,'text':text,'target_property':prop,'target_value':val,'target_year':year}

def generate_queries_for_table(t):
    p=t['properties']; out=[]; n=0
    if p['strictly_increasing']:
        out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'trend',random.choice(INC),'strictly_increasing',True)); n+=1
    elif p['strictly_decreasing']:
        out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'trend',random.choice(DEC),'strictly_decreasing',True)); n+=1
    out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'maximum',random.choice(MAXT).format(year=p['max_year']),'max_year',year=p['max_year'])); n+=1
    out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'minimum',random.choice(MINT).format(year=p['min_year']),'min_year',year=p['min_year'])); n+=1
    i=random.randrange(len(t['years'])); y,v=t['years'][i],t['values'][i]
    out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'exact_value',random.choice(EXACT).format(year=y,value=v),'exact_value',v,y)); n+=1
    if t['values'][-1]>t['values'][0]: out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'endpoint_comparison',random.choice(ENDHI),'end_greater_than_start',True))
    elif t['values'][-1]<t['values'][0]: out.append(qrec(f"{t['table_id']}_q{n}",t['table_id'],'endpoint_comparison',random.choice(ENDLO),'end_less_than_start',True))
    return out

def query_is_satisfied(q,t):
    p=t['properties']; qt=q['query_type']
    if qt=='trend': return p[q['target_property']]==q['target_value']
    if qt=='maximum': return p['max_year']==q['target_year']
    if qt=='minimum': return p['min_year']==q['target_year']
    if qt=='exact_value':
        if q['target_year'] not in t['years']: return False
        return t['values'][t['years'].index(q['target_year'])]==q['target_value']
    if qt=='endpoint_comparison':
        a,b=t['values'][0],t['values'][-1]
        return b>a if q['target_property']=='end_greater_than_start' else b<a
    return False


## Stage 6 — Query-conditioned minimal counterfactual generator


In [7]:
def rebuild_counterfactual(source, values, q):
    cf={'table_id':source['table_id']+'_'+q['query_id'].split('_')[-1]+'_cf','source_table_id':source['table_id'],
        'years':source['years'].copy(),'values':[int(x) for x in values],'pattern':'counterfactual','negative_type':'query_counterfactual','query_id':q['query_id']}
    cf['properties']=extract_properties(cf); return cf

def valid_value(x): return 5 <= x <= 100

def generate_minimal_counterfactual(table,q):
    v=table['values'].copy(); candidates=[]; qt=q['query_type']
    # Search one-coordinate integer edits by increasing L1 distance. This yields a minimal edit in the search space.
    for delta in range(1,96):
        for i in range(len(v)):
            for sign in (-1,1):
                nv=v.copy(); nv[i]=v[i]+sign*delta
                if not valid_value(nv[i]): continue
                cf=rebuild_counterfactual(table,nv,q)
                if not query_is_satisfied(q,cf):
                    candidates.append((delta,i,cf))
        if candidates:
            candidates.sort(key=lambda x:(x[0],x[1]))
            cf=candidates[0][2]
            cf['edit_l1']=data_difference(table['values'],cf['values'])*len(v)
            cf['changed_indices']=[i for i,(a,b) in enumerate(zip(v,cf['values'])) if a!=b]
            return cf
    return None

# Validate the central invariant: original satisfies Q; counterfactual does not.
for i in range(50):
    t=generate_table(f'cfcheck_{i}')
    for q in generate_queries_for_table(t):
        assert query_is_satisfied(q,t)
        cf=generate_minimal_counterfactual(t,q)
        if cf is not None: assert not query_is_satisfied(q,cf)
print('Counterfactual sanity checks passed.')


Counterfactual sanity checks passed.


## Stage 7 — Assemble dataset and leakage-safe table-level splits


In [8]:
def build_dataset(n_tables=600, charts_per_table=4):
    tables=[]; charts=[]; queries=[]; shuffled=[]; counterfactuals=[]; cf_charts=[]
    for i in tqdm(range(n_tables)):
        t=generate_table(f'table_{i:05d}'); tables.append(t)
        ch=generate_charts_for_table(t,charts_per_table); charts.extend(ch)
        s=create_shuffled_negative(t); shuffled.append(s)
        qs=generate_queries_for_table(t); queries.extend(qs)
        for q in qs:
            cf=generate_minimal_counterfactual(t,q)
            if cf is not None:
                counterfactuals.append(cf)
                # Render CF in the same style as the first source chart for controlled comparison.
                src=ch[0]; cid=cf['table_id']+'_chart'; path=PROJECT_ROOT/'data/counterfactuals'/f'{cid}.png'
                render_chart(cf,src['chart_type'],src['color'],path)
                cf_charts.append({'chart_id':cid,'table_id':cf['table_id'],'source_table_id':t['table_id'],'query_id':q['query_id'],
                                  'chart_type':src['chart_type'],'color':src['color'],'path':str(path),'negative_type':'query_counterfactual'})
    ids=[t['table_id'] for t in tables]; random.Random(SEED).shuffle(ids)
    a=int(.8*len(ids)); b=int(.9*len(ids)); split={x:'train' for x in ids[:a]}; split.update({x:'val' for x in ids[a:b]}); split.update({x:'test' for x in ids[b:]})
    for t in tables: t['split']=split[t['table_id']]
    for c in charts: c['split']=split[c['table_id']]
    for q in queries: q['split']=split[q['table_id']]
    for x in shuffled: x['split']=split[x['source_table_id']]
    for x in counterfactuals: x['split']=split[x['source_table_id']]
    for x in cf_charts: x['split']=split[x['source_table_id']]
    return {'tables':tables,'charts':charts,'queries':queries,'shuffled':shuffled,'counterfactuals':counterfactuals,'cf_charts':cf_charts}

DATA=build_dataset(n_tables=600)
{k:len(v) for k,v in DATA.items()}


  0%|          | 0/600 [00:00<?, ?it/s]

{'tables': 600,
 'charts': 2400,
 'queries': 2574,
 'shuffled': 600,
 'counterfactuals': 2574,
 'cf_charts': 2574}

In [9]:
# Save metadata locally in /content # will change it !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
for name,records in DATA.items():
    path=PROJECT_ROOT/'data/metadata'/f'{name}.json'
    with open(path,'w') as f: json.dump(records,f,indent=2)
print('Saved metadata.')


Saved metadata.


## Stage 8 — Frozen CLIP baseline


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME='openai/clip-vit-base-patch32'
processor=CLIPProcessor.from_pretrained(MODEL_NAME)
clip=CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip.eval()
for p in clip.parameters(): p.requires_grad=False
print('device:',DEVICE)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

device: cpu


In [11]:
@torch.no_grad()
def encode_images(paths, batch_size=32):
    outs = []

    for i in tqdm(range(0, len(paths), batch_size)):
        ims = [
            Image.open(p).convert("RGB")
            for p in paths[i:i + batch_size]
        ]

        x = processor(
            images=ims,
            return_tensors="pt"
        )

        pixel_values = x["pixel_values"].to(DEVICE)

        # Vision encoder
        output = clip.vision_model(
            pixel_values=pixel_values
        )

        # Extract actual tensor
        pooled = output.pooler_output

        # Project to CLIP's 512-dimensional shared space
        z = clip.visual_projection(pooled)

        z = F.normalize(z, dim=-1)

        outs.append(z.cpu())

    return torch.cat(outs, dim=0)


@torch.no_grad()
def encode_texts(texts, batch_size=64):
    outs = []

    for i in tqdm(range(0, len(texts), batch_size)):
        x = processor(
            text=texts[i:i + batch_size],
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        input_ids = x["input_ids"].to(DEVICE)
        attention_mask = x["attention_mask"].to(DEVICE)

        # Text encoder
        output = clip.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract actual tensor
        pooled = output.pooler_output

        # Project to CLIP's 512-dimensional shared space
        z = clip.text_projection(pooled)

        z = F.normalize(z, dim=-1)

        outs.append(z.cpu())

    return torch.cat(outs, dim=0)

## Stage 9 — Data-aware projection model


In [12]:
EMB=clip.config.projection_dim
class Projector(nn.Module):
    def __init__(self,d=EMB,h=512,out=256):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d,h),nn.GELU(),nn.Linear(h,out))
    def forward(self,x): return F.normalize(self.net(x),dim=-1)

class DataAwareRetriever(nn.Module):
    def __init__(self):
        super().__init__(); self.image_projector=Projector(); self.text_projector=Projector()
    def image(self,x): return self.image_projector(x)
    def text(self,x): return self.text_projector(x)

model=DataAwareRetriever().to(DEVICE)


## Stages 10–13 — Multi-task contrastive + counterfactual-aware training


In [13]:
# Precompute frozen CLIP features for training records.
table_by_id={t['table_id']:t for t in DATA['tables']}
charts_by_table=defaultdict(list)
for c in DATA['charts']: charts_by_table[c['table_id']].append(c)
queries_by_table=defaultdict(list)
for q in DATA['queries']: queries_by_table[q['table_id']].append(q)
cf_by_query={c['query_id']:c for c in DATA['cf_charts']}

train_tables=[t for t in DATA['tables'] if t['split']=='train']
train_chart_records=[c for c in DATA['charts'] if c['split']=='train']
train_paths=[c['path'] for c in train_chart_records]
train_img_feats=encode_images(train_paths)
chart_feat={c['chart_id']:z for c,z in zip(train_chart_records,train_img_feats)}

# Counterfactual image features
train_cf=[c for c in DATA['cf_charts'] if c['split']=='train']
if train_cf:
    zcf=encode_images([c['path'] for c in train_cf]); cf_feat={c['query_id']:z for c,z in zip(train_cf,zcf)}
else: cf_feat={}

# Text features for table serializations and queries
train_table_texts=[table_text(t) for t in train_tables]
ztt=encode_texts(train_table_texts); table_feat={t['table_id']:z for t,z in zip(train_tables,ztt)}
train_queries=[q for q in DATA['queries'] if q['split']=='train']
zq=encode_texts([q['text'] for q in train_queries]); query_feat={q['query_id']:z for q,z in zip(train_queries,zq)}
print(len(chart_feat),len(table_feat),len(query_feat),len(cf_feat))


  0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/33 [00:00<?, ?it/s]

1920 480 2067 2067


In [14]:
def triplet_margin(anchor,pos,neg,margin=.2):
    sp=(anchor*pos).sum(-1); sn=(anchor*neg).sum(-1)
    return F.relu(margin-sp+sn).mean()

def sample_batch(batch_size=64):
    usable=[t for t in train_tables if len(charts_by_table[t['table_id']])>=2 and queries_by_table[t['table_id']]]
    return random.sample(usable,min(batch_size,len(usable)))

optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)

def train_epoch(batch_size=64,lambda_chart=1.,lambda_table=1.,lambda_query=1.,lambda_cf=1.):
    model.train(); random.shuffle(train_tables); losses=[]
    steps=max(1,len(train_tables)//batch_size)
    for _ in range(steps):
        batch=sample_batch(batch_size); Lc=[]; Lt=[]; Lq=[]; Lcf=[]
        for t in batch:
            tid=t['table_id']; cs=charts_by_table[tid]; a,p=random.sample(cs,2)
            # random other-table chart as standard negative
            other=random.choice([x for x in train_tables if x['table_id']!=tid]); n=random.choice(charts_by_table[other['table_id']])
            za=model.image(chart_feat[a['chart_id']].to(DEVICE)); zp=model.image(chart_feat[p['chart_id']].to(DEVICE)); zn=model.image(chart_feat[n['chart_id']].to(DEVICE))
            Lc.append(triplet_margin(za[None],zp[None],zn[None]))
            zt=model.text(table_feat[tid].to(DEVICE)); ztn=model.text(table_feat[other['table_id']].to(DEVICE))
            Lt.append(triplet_margin(za[None],zt[None],ztn[None]))
            q=random.choice(queries_by_table[tid]); zqv=model.text(query_feat[q['query_id']].to(DEVICE))
            Lq.append(triplet_margin(zqv[None],za[None],zn[None]))
            if q['query_id'] in cf_feat:
                zcfv=model.image(cf_feat[q['query_id']].to(DEVICE)); Lcf.append(triplet_margin(zqv[None],za[None],zcfv[None],margin=.1))
        loss=lambda_chart*torch.stack(Lc).mean()+lambda_table*torch.stack(Lt).mean()+lambda_query*torch.stack(Lq).mean()
        if Lcf: loss=loss+lambda_cf*torch.stack(Lcf).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(loss.item())
    return float(np.mean(losses))

for epoch in range(5):
    loss=train_epoch()
    print(f'Epoch {epoch+1}: {loss:.4f}')
torch.save(model.state_dict(),PROJECT_ROOT/'models/data_aware_retriever.pt')


Epoch 1: 0.6749
Epoch 2: 0.6318
Epoch 3: 0.6199
Epoch 4: 0.4865
Epoch 5: 0.3854


## Stage 14 — Optional data/style disentanglement experiment


In [15]:
class DisentangledImageProjector(nn.Module):
    def __init__(self,d=EMB,h=512,out=256,style_dim=64):
        super().__init__(); self.shared=nn.Sequential(nn.Linear(d,h),nn.GELU())
        self.data_head=nn.Linear(h,out); self.style_head=nn.Linear(h,style_dim)
    def forward(self,x):
        h=self.shared(x)
        return F.normalize(self.data_head(h),dim=-1), F.normalize(self.style_head(h),dim=-1)

# This module is intentionally kept as an ablation rather than replacing the main model.
# Train a style classifier/probe on style_head and test whether data_head has low style predictability.
disentangled_projector=DisentangledImageProjector().to(DEVICE)
print('Optional ablation initialized.')


Optional ablation initialized.


## Stage 15 — Retrieval engine


In [16]:
@torch.no_grad()
def projected_test_embeddings(split='test'):
    model.eval()
    charts=[c for c in DATA['charts'] if c['split']==split]
    qs=[q for q in DATA['queries'] if q['split']==split]
    zi=encode_images([c['path'] for c in charts]).to(DEVICE)
    zq=encode_texts([q['text'] for q in qs]).to(DEVICE)
    return charts,qs,model.image(zi).cpu(),model.text(zq).cpu()

@torch.no_grad()
def retrieve_charts(query_text,chart_records,chart_embeddings,k=5):
    z=encode_texts([query_text]).to(DEVICE); z=model.text(z).cpu()[0]
    scores=chart_embeddings@z
    idx=torch.topk(scores,min(k,len(scores))).indices.tolist()
    return [(chart_records[i],float(scores[i])) for i in idx]


## Stage 16 — Evaluation: Recall@K, MRR, style/data gap, counterfactual accuracy


In [17]:
def retrieval_metrics(charts,queries,Zc,Zq,ks=(1,5,10)):
    scores=Zq@Zc.T; recalls={k:[] for k in ks}; rr=[]
    for i,q in enumerate(queries):
        order=torch.argsort(scores[i],descending=True).tolist()
        relevant={j for j,c in enumerate(charts) if c['table_id']==q['table_id']}
        rank=min(order.index(j)+1 for j in relevant)
        rr.append(1/rank)
        for k in ks: recalls[k].append(any(j in relevant for j in order[:k]))
    out={f'Recall@{k}':float(np.mean(v)) for k,v in recalls.items()}; out['MRR']=float(np.mean(rr)); return out

charts_t,queries_t,Zc_t,Zq_t=projected_test_embeddings('test')
metrics=retrieval_metrics(charts_t,queries_t,Zc_t,Zq_t)
print(metrics)


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

{'Recall@1': 0.03137254901960784, 'Recall@5': 0.10980392156862745, 'Recall@10': 0.1843137254901961, 'MRR': 0.09016549169854197}


In [18]:
@torch.no_grad()
def style_data_gap(split='test',max_groups=100):
    model.eval(); ts=[t for t in DATA['tables'] if t['split']==split][:max_groups]
    same_data=[]; diff_data_same_style=[]
    # cache selected chart embeddings
    recs=[]
    for t in ts: recs += charts_by_table[t['table_id']]
    # charts_by_table contains all splits; select existing records
    z=encode_images([r['path'] for r in recs]).to(DEVICE); z=model.image(z).cpu(); emb={r['chart_id']:e for r,e in zip(recs,z)}
    for t in ts:
        cs=charts_by_table[t['table_id']]
        if len(cs)>=2: same_data.append(float(emb[cs[0]['chart_id']]@emb[cs[1]['chart_id']]))
    # compare matching style across different tables when available
    for i in range(len(ts)-1):
        a=charts_by_table[ts[i]['table_id']]; b=charts_by_table[ts[i+1]['table_id']]
        pair=next(((x,y) for x in a for y in b if x['chart_type']==y['chart_type'] and x['color']==y['color']),None)
        if pair: diff_data_same_style.append(float(emb[pair[0]['chart_id']]@emb[pair[1]['chart_id']]))
    return {'same_data_diff_style':np.mean(same_data),'diff_data_same_style':np.mean(diff_data_same_style),
            'gap':np.mean(same_data)-np.mean(diff_data_same_style)}

print(style_data_gap())


  0%|          | 0/8 [00:00<?, ?it/s]

{'same_data_diff_style': np.float64(0.8111222594976425), 'diff_data_same_style': np.float64(0.4551997536368537), 'gap': np.float64(0.3559225058607888)}


In [19]:
@torch.no_grad()
def counterfactual_accuracy(split='test',limit=300):
    model.eval(); cfs=[c for c in DATA['cf_charts'] if c['split']==split][:limit]
    correct=[]
    for cf in tqdm(cfs):
        q=next(q for q in DATA['queries'] if q['query_id']==cf['query_id'])
        pos=random.choice(charts_by_table[q['table_id']])
        zi=encode_images([pos['path'],cf['path']]).to(DEVICE); zi=model.image(zi).cpu()
        zq=model.text(encode_texts([q['text']]).to(DEVICE)).cpu()[0]
        correct.append(float(zi[0]@zq)>float(zi[1]@zq))
    return float(np.mean(correct)) if correct else float('nan')

print('Counterfactual pair accuracy:',counterfactual_accuracy())


  0%|          | 0/255 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

... (502 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Counterfactual pair accuracy: 0.796078431372549


## Stage 17 — Ablation and robustness experiments


In [20]:
# Recommended controlled runs. Reinitialize the model before each run and record metrics.
ABLATIONS = {
    'A0_frozen_CLIP': {'train': False},
    'A1_standard_contrastive': {'lambda_chart':1,'lambda_table':1,'lambda_query':1,'lambda_cf':0},
    'A2_plus_counterfactuals': {'lambda_chart':1,'lambda_table':1,'lambda_query':1,'lambda_cf':1},
    'A3_counterfactual_heavy': {'lambda_chart':1,'lambda_table':1,'lambda_query':1,'lambda_cf':2},
}
print(pd.DataFrame(ABLATIONS).T)

# Robustness extensions to add after the core result is stable:
# 1) train bar -> test line
# 2) held-out colors
# 3) fonts/grid/marker/layout shifts
# 4) performance vs counterfactual edit distance
# 5) query-type breakdown: trend/max/min/exact/endpoint


                         train lambda_chart lambda_table lambda_query  \
A0_frozen_CLIP           False          NaN          NaN          NaN   
A1_standard_contrastive    NaN          1.0          1.0          1.0   
A2_plus_counterfactuals    NaN          1.0          1.0          1.0   
A3_counterfactual_heavy    NaN          1.0          1.0          1.0   

                        lambda_cf  
A0_frozen_CLIP                NaN  
A1_standard_contrastive       0.0  
A2_plus_counterfactuals       1.0  
A3_counterfactual_heavy       2.0  


## Stage 18 — Demo + export results


In [23]:
def show_retrieval(query_text, k=5):
    results = retrieve_charts(
        query_text,
        charts_t,
        Zc_t,
        k
    )

    fig, axes = plt.subplots(
        1,
        len(results),
        figsize=(4 * len(results), 4)
    )

    # Handle a single result
    if len(results) == 1:
        axes = [axes]

    for ax, (rec, score) in zip(axes, results):
        img = Image.open(
            rec["path"]
        ).convert("RGB")

        ax.imshow(img)
        ax.axis("off")

        ax.set_title(
            f"{score:.3f}\n"
            f"{rec['table_id']}"
        )

    plt.suptitle(query_text)
    plt.tight_layout()
    plt.show()

    return results


# Example:
# show_retrieval(
#     "Which chart shows a steady increase over time?",
#     k=5
# )

In [24]:
results_summary={'main_metrics':metrics,'style_data_gap':style_data_gap(),'counterfactual_accuracy':counterfactual_accuracy(limit=100)}
with open(PROJECT_ROOT/'results/summary.json','w') as f: json.dump(results_summary,f,indent=2)
print(json.dumps(results_summary,indent=2))

# Zip the local project if you want to download it before the Colab runtime resets.
shutil.make_archive('/content/data_aware_chart_retrieval','zip',PROJECT_ROOT)
print('/content/data_aware_chart_retrieval.zip')


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

... (193 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{
  "main_metrics": {
    "Recall@1": 0.03137254901960784,
    "Recall@5": 0.10980392156862745,
    "Recall@10": 0.1843137254901961,
    "MRR": 0.09016549169854197
  },
  "style_data_gap": {
    "same_data_diff_style": 0.8111222594976425,
    "diff_data_same_style": 0.4551997536368537,
    "gap": 0.3559225058607888
  },
  "counterfactual_accuracy": 0.77
}
/content/data_aware_chart_retrieval.zip
